In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
%load_ext autoreload
%autoreload 2

from cse250a.modules.config import config

In [ ]:
df = pd.read_csv(config.paths.air_quality_csv, sep=";", decimal=",")
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]   # two empty trailing cols
df = df.dropna(how="all")                                # blank trailing rows
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H.%M.%S")
df = df.drop(columns=["Date", "Time"]).set_index("datetime")
df = df.replace(-200, np.nan)    

**Each gas vs. its nominal sensor.** CO is fit best on linear axes (R² 0.773 vs 0.717 log-log); the
horizontal stripes in its log-log panel come from the analyzer's 0.1 mg/m³ resolution. NMHC and NOx
straighten out on log-log (0.770 → 0.890 and 0.430 → 0.615); S3 *decreases* as NOx rises (tungsten-oxide
response is inverse). NO2 is essentially unrelated to its nominal sensor S4 (R² ≈ 0.03 either way),
so S4 alone cannot stand in for NO2 — consistent with the cross-sensitivity findings.

In [ ]:
# %%
PAIRS = [("CO(GT)", "PT08.S1(CO)"), ("NMHC(GT)", "PT08.S2(NMHC)"),
         ("NOx(GT)", "PT08.S3(NOx)"), ("NO2(GT)", "PT08.S4(NO2)")]

def line_fit(x, y):
    """Least-squares line y = k x + b; returns (r2, slope, intercept)."""
    k, b = np.polyfit(x, y, 1)
    return np.corrcoef(x, y)[0, 1] ** 2, k, b

rows = []
fig, axes = plt.subplots(2, len(PAIRS), figsize=(4 * len(PAIRS), 7))
for j, (g, s) in enumerate(PAIRS):
    m = df[[g, s]].dropna()
    m = m[m[g] > 0]                                    # log needs y > 0
    x, y = m[s].to_numpy(), m[g].to_numpy()

    views = [(axes[0, j], x, y, s, g),
             (axes[1, j], np.log(x), np.log(y), f"log({s})", f"log({g})")]
    fits = []
    for ax, xv, yv, xl, yl in views:
        r2, k, b = line_fit(xv, yv)
        fits.append(r2)
        ax.scatter(xv, yv, s=2, alpha=0.2)
        grid = np.linspace(xv.min(), xv.max(), 100)
        ax.plot(grid, k * grid + b, color="C3", lw=1.5, label=f"slope={k:.3g}, R²={r2:.3f}")
        ax.set_xlabel(xl, fontsize=8); ax.set_ylabel(yl, fontsize=8)
        ax.legend(loc="upper left", fontsize=8)
    axes[0, j].set_title(f"{g} vs {s}", fontsize=10)

    rows.append({"gas": g, "sensor": s, "n": len(m),
                 "spearman ρ": round(m[g].corr(m[s], method="spearman"), 3),
                 "R² linear": round(fits[0], 3), "R² log-log": round(fits[1], 3)})

fig.text(0.0, 0.73, "linear", rotation=90, va="center", fontsize=11)
fig.text(0.0, 0.27, "log-log", rotation=90, va="center", fontsize=11)
fig.suptitle("Each GT gas vs. its nominal sensor: linear and log-log fits")
plt.tight_layout(rect=(0.015, 0, 1, 1))
plt.savefig(config.paths.figure("kiernan", "nominal_sensor_fits.png"), dpi=160, bbox_inches="tight")
plt.show()

pd.DataFrame(rows).set_index("gas")



In [ ]:
# %%
m = df[["NO2(GT)", "PT08.S4(NO2)", "T"]].dropna()
fig, ax = plt.subplots(figsize=(6, 4.5))
sc = ax.scatter(m["PT08.S4(NO2)"], m["NO2(GT)"], c=m["T"], cmap="coolwarm", s=3, alpha=0.4)
fig.colorbar(sc, label="T (°C)")
ax.set_xlabel("PT08.S4(NO2) (raw)"); ax.set_ylabel("NO2(GT) (µg/m³)")
ax.set_title("NO2 vs. S4, colored by temperature")
plt.savefig(config.paths.figure("kiernan", "no2_s4_by_temperature.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# %%
def ols(target, features, data):
    """Fit target = θ1 + Σ θi·feature_i by least squares; return (θ series, R², RMSE)."""
    m = data[[target] + features].dropna()
    A = np.column_stack([np.ones(len(m)), m[features].to_numpy()])
    y = m[target].to_numpy()
    theta, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - A @ theta
    r2 = 1 - resid.var() / y.var()
    rmse = np.sqrt((resid ** 2).mean())
    return pd.Series(theta, index=["intercept"] + features), r2, rmse

SENSORS = ["PT08.S1(CO)", "PT08.S2(NMHC)", "PT08.S3(NOx)", "PT08.S4(NO2)", "PT08.S5(O3)"]
MODELS = {
    "S4":                ["PT08.S4(NO2)"],
    "S4 + T + RH":       ["PT08.S4(NO2)", "T", "RH"],
    "S4 + T + AH":       ["PT08.S4(NO2)", "T", "AH"],
    "S4 + T + RH + AH":  ["PT08.S4(NO2)", "T", "RH", "AH"],
    "all sensors + weather": SENSORS + ["T", "RH", "AH"],
}

# same rows for every model so R² values are comparable
common = df[["NO2(GT)"] + SENSORS + ["T", "RH", "AH"]].dropna()

rows = []
for name, feats in MODELS.items():
    theta, r2, rmse = ols("NO2(GT)", feats, common)
    rows.append({"model": name, "R²": round(r2, 3), "RMSE": round(rmse, 1)})
    if name == "S4 + T + RH":
        print("NO2(GT) ≈", " + ".join(f"{v:.3g}·{k}" if k != "intercept" else f"{v:.4g}"
                                       for k, v in theta.items()))
pd.DataFrame(rows).set_index("model")

In [ ]:
# %%
logged = common.copy()
for c in ["NO2(GT)"] + SENSORS:
    logged[c] = np.log(common[c])

rows = []
for name, feats in MODELS.items():
    theta, r2, _ = ols("NO2(GT)", feats, logged)
    A = np.column_stack([np.ones(len(logged)), logged[feats].to_numpy()])
    pred = np.exp(A @ theta.to_numpy())                       # back to µg/m³
    rmse = np.sqrt(((common["NO2(GT)"] - pred) ** 2).mean())
    rows.append({"model": f"log {name}", "R² (log scale)": round(r2, 3), "RMSE µg/m³": round(rmse, 1)})
pd.DataFrame(rows).set_index("model")

In [ ]:
# %%
SEASON = {12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring",
          6: "summer", 7: "summer", 8: "summer", 9: "autumn", 10: "autumn", 11: "autumn"}
SEASONS = ["winter", "spring", "summer", "autumn"]

season = pd.Categorical(common.index.month.map(SEASON), SEASONS)
onehot = pd.get_dummies(season, prefix="season", dtype=float).set_index(common.index)
feat = common.join(onehot)
ONEHOT = list(onehot.columns)

def fit_r2(features, data, target="NO2(GT)"):
    """Least squares with NO intercept column: the full one-hot set plays that role."""
    A, y = data[features].to_numpy(), data[target].to_numpy()
    theta, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - A @ theta
    return pd.Series(theta, index=features), 1 - resid.var() / y.var()

MODELS_S = {
    "season only":          ONEHOT,
    "S4 + season":          ["PT08.S4(NO2)"] + ONEHOT,
    "S4 + T + RH + season": ["PT08.S4(NO2)", "T", "RH"] + ONEHOT,
}

rows, thetas = [], {}
for name, feats in MODELS_S.items():
    thetas[name], r2 = fit_r2(feats, feat)
    rows.append({"model": name, "R²": round(r2, 3)})

display(onehot.sum().astype(int).rename("hours"))
display(pd.DataFrame(rows).set_index("model"))
thetas["S4 + T + RH + season"].round(3)

In [ ]:
# %%
SEASON = {12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring",
          6: "summer", 7: "summer", 8: "summer", 9: "autumn", 10: "autumn", 11: "autumn"}
COLORS = {"winter": "#3B6FB6", "spring": "#4C9A2A", "summer": "#E1A100", "autumn": "#C0504D"}

m = df[["NO2(GT)", "PT08.S4(NO2)"]].dropna()
m["season"] = m.index.month.map(SEASON)
x_all, y_all = m["PT08.S4(NO2)"].to_numpy(), m["NO2(GT)"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)

# left: everything pooled, one line
k, b = np.polyfit(x_all, y_all, 1)
r2 = np.corrcoef(x_all, y_all)[0, 1] ** 2
axes[0].scatter(x_all, y_all, s=2, alpha=0.2, color="grey")
grid = np.linspace(x_all.min(), x_all.max(), 100)
axes[0].plot(grid, k * grid + b, color="black", lw=2, label=f"all hours: slope={k:.3f}, R²={r2:.3f}")
axes[0].set_title("Pooled: S4 looks unrelated to NO2")

# right: one colour and one line per season
for name in COLORS:
    d = m[m["season"] == name]
    x, y = d["PT08.S4(NO2)"].to_numpy(), d["NO2(GT)"].to_numpy()
    k, b = np.polyfit(x, y, 1)
    r2 = np.corrcoef(x, y)[0, 1] ** 2
    axes[1].scatter(x, y, s=2, alpha=0.25, color=COLORS[name])
    grid = np.linspace(x.min(), x.max(), 100)
    axes[1].plot(grid, k * grid + b, color=COLORS[name], lw=2,
                 label=f"{name}: slope={k:.3f}, R²={r2:.2f}")
axes[1].set_title("By season: S4 tracks NO2 within each season")

for ax in axes:
    ax.set_xlabel("PT08.S4(NO2) (raw)")
    ax.legend(loc="upper left", fontsize=8, markerscale=4)
axes[0].set_ylabel("NO2(GT) (µg/m³)")
fig.suptitle("NO2(GT) vs. its nominal sensor S4")
plt.tight_layout()
plt.savefig(config.paths.figure("kiernan", "no2_s4_by_season.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# %%
def ols_report(target, features, data):
    """Least squares fit; prints the equation, R², RMSE and standardized coefficients."""
    m = data[[target] + features].dropna()
    A = np.column_stack([np.ones(len(m)), m[features].to_numpy()])
    y = m[target].to_numpy()
    theta, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - A @ theta
    r2 = 1 - resid.var() / y.var()
    rmse = np.sqrt((resid ** 2).mean())
    std_coef = theta[1:] * m[features].std().to_numpy() / m[target].std()
    terms = " + ".join(f"{t:.3g}·{f}" for t, f in zip(theta[1:], features))
    print(f"{target} ≈ {theta[0]:.4g} + {terms}")
    print(f"  n={len(m)}  R²={r2:.3f}  RMSE={rmse:.1f}")
    return pd.Series(std_coef, index=features, name="standardized").round(2)

cols = ["NO2(GT)", "PT08.S4(NO2)", "PT08.S5(O3)", "T", "AH"]
no2_rows = df[cols].dropna()                    # same rows for every model

In [ ]:
MODELS = [
    ["T", "AH"],
    ["PT08.S4(NO2)", "T", "AH"],
    ["PT08.S4(NO2)", "PT08.S5(O3)", "T", "AH"],
]

# %%
# Linear: NO2 = θ1 + θ·features
for feats in MODELS:
    display(ols_report("NO2(GT)", feats, no2_rows))

In [ ]:
# %%
# Log: log(NO2) = θ1 + θ·features, with NO2, S4 and S5 logged (T and AH left as-is)
log_rows = no2_rows.copy()
for c in ["NO2(GT)", "PT08.S4(NO2)", "PT08.S5(O3)"]:
    log_rows[c] = np.log(no2_rows[c])

rows = []
for feats in MODELS:
    A = np.column_stack([np.ones(len(log_rows)), log_rows[feats].to_numpy()])
    theta, *_ = np.linalg.lstsq(A, log_rows["NO2(GT)"].to_numpy(), rcond=None)
    pred_log = A @ theta
    pred = np.exp(pred_log)                     # back to µg/m³ to compare with linear
    y, ly = no2_rows["NO2(GT)"], log_rows["NO2(GT)"]
    rows.append({"features": " + ".join(feats),
                 "R² (log scale)": round(1 - ((ly - pred_log) ** 2).mean() / ly.var(), 3),
                 "R² (back in µg/m³)": round(1 - ((y - pred) ** 2).mean() / y.var(), 3),
                 "RMSE µg/m³": round(np.sqrt(((y - pred) ** 2).mean()), 1)})
pd.DataFrame(rows).set_index("features")

In [ ]:
# %%
def fit_predict(data, feats, target="NO2(GT)"):
    """Least squares fit; returns predictions for every row."""
    A = np.column_stack([np.ones(len(data)), data[feats].to_numpy()])
    theta, *_ = np.linalg.lstsq(A, data[target].to_numpy(), rcond=None)
    return A @ theta

y = no2_rows["NO2(GT)"].to_numpy()
lim = (0, y.max() * 1.05)
SHORT = {"PT08.S4(NO2)": "S4", "PT08.S5(O3)": "S5"}

fig, axes = plt.subplots(2, len(MODELS), figsize=(4.2 * len(MODELS), 8), sharex=True, sharey=True)
for j, feats in enumerate(MODELS):
    name = " + ".join(SHORT.get(f, f) for f in feats)
    for i, (label, pred) in enumerate([
        ("linear", fit_predict(no2_rows, feats)),
        ("log",    np.exp(fit_predict(log_rows, feats))),     # back to µg/m³
    ]):
        r2 = 1 - ((y - pred) ** 2).mean() / y.var()
        rmse = np.sqrt(((y - pred) ** 2).mean())
        ax = axes[i, j]
        ax.scatter(y, pred, s=2, alpha=0.2)
        ax.plot(lim, lim, color="C3", lw=1.2, label="perfect prediction")
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_title(f"{label}: {name}", fontsize=10)
        ax.text(0.04, 0.96, f"R² = {r2:.3f}\nRMSE = {rmse:.1f}", transform=ax.transAxes,
                va="top", fontsize=9, bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
for ax in axes[-1]:
    ax.set_xlabel("actual NO2(GT) (µg/m³)")
for ax in axes[:, 0]:
    ax.set_ylabel("predicted NO2 (µg/m³)")
axes[0, 0].legend(loc="lower right", fontsize=8)
fig.suptitle(f"Explaining NO2 from its nominal sensor + weather (n = {len(y)} hours)")
plt.tight_layout()
plt.savefig(config.paths.figure("kiernan", "no2_nominal_models.png"), dpi=160, bbox_inches="tight")
plt.show()